# 4-9 自訂模型評分的評估

> 📚 **第 4 單元：提示詞評估**｜第 9 課 / 共 9 課
> ⏱️ 預估時間：40 分鐘

## 🎯 這一課你會學到

- 寫一個**自訂的模型評分函式**：用 Python 呼叫 Claude，依照你設計的評分準則打分數
- 設計包含**多個標準（簡潔度、正確性、語氣）**和**評分範例**的評分提示詞
- 從檔案載入很長的測試資料（`file://articles/article1.txt`）

> 📁 **這一課的檔案已經準備好了**：和這份講義同一個資料夾裡有 `promptfooconfig.yaml`、`prompts.py`、`custom_llm_eval.py` 和 `articles/` 資料夾（8 篇維基百科文章）。
>
> 🔄 **2026 更新**：
> - 產生摘要的模型：`claude-3-5-sonnet-20240620` → `claude-sonnet-4-6`（設定檔的 label 也改成 "Sonnet 4.6"）。
> - **評分用的模型**（在 `custom_llm_eval.py` 裡）：原本也是 `claude-3-5-sonnet-20240620`，但這段程式用了「**預先填寫回答（prefill）**」技巧（assistant 訊息先寫好 `<json>`），而 Sonnet 4.6 **不支援** prefill，所以新手版改用支援 prefill 的 **`claude-haiku-4-5`** 當評分模型。
> - 講義中的程式碼範例保留原課程的寫法，實際執行請用資料夾裡已更新的檔案。
> - ⚠️ 這一課會把 8 篇很長的文章送給模型好幾次（3 個提示詞 × 8 篇文章，再加上評分），**費用會比前幾課高一些**（大約幾十美分到一兩美元）。

---

**注意：這一課所在的資料夾裡有相關的程式碼檔案。如果你想跟著執行評估，請下載整個資料夾。**

在這一課，我們會看看怎麼用 promptfoo 撰寫**自訂的模型評分評估**。我們從一個簡單的提示詞目標開始：我們想寫一個提示詞，能把又長又技術性的**維基百科文章**，轉換成**適合小學生**閱讀的簡短摘要。

舉例來說，給定整篇[關於卷積神經網路的維基百科條目](https://en.wikipedia.org/wiki/Convolutional_neural_network)，我們希望得到像這樣簡單的輸出摘要：

> Convolutional neural networks, or CNNs, are a special type of computer program that can learn to recognize images and patterns. They work a bit like the human brain, using layers of artificial "neurons" to process information.
CNNs are really good at tasks like identifying objects in pictures or recognizing faces. They do this by breaking down images into smaller pieces and looking for important features, kind of like putting together a puzzle.
What makes CNNs special is that they can learn these features on their own by looking at lots of examples. This allows them to get better and better at recognizing things, sometimes even matching human-level performance.
Scientists and engineers use CNNs for all sorts of cool applications, like helping self-driving cars see the road, finding new medicines, or even teaching computers to play games like chess and Go.

**翻譯**：「卷積神經網路，簡稱 CNN，是一種特別的電腦程式，能學會辨認圖片和圖案。它們的運作方式有點像人腦，用一層一層人造的『神經元』來處理資訊。CNN 非常擅長辨認圖片裡的東西或認出人臉。它們的做法是把圖片拆成一小塊一小塊，尋找重要的特徵，有點像在拼拼圖。CNN 特別的地方在於，它們可以透過看很多例子，自己學會這些特徵。這讓它們辨認東西的能力越來越好，有時候甚至能和人類一樣厲害。科學家和工程師把 CNN 用在各種很酷的地方，例如幫助自駕車看清楚道路、尋找新藥，甚至教電腦下西洋棋和圍棋。」

為了評估提示詞的效果，我們要寫一個**自訂的模型評分斷言**，從三個指標評估產生的摘要：

* **簡潔度（Conciseness，1～5 分）**：摘要是不是已經盡可能簡潔了？
* **正確性（Accuracy，1～5 分）**：根據原本的文章，摘要是不是完全正確？
* **語氣（Tone，1～5 分）**：摘要是否適合沒有任何技術背景的小學生？

每個指標都會得到一個 1 到 5 之間的分數。我們會把它們平均起來，目標是**平均至少 4.5 分（滿分 5 分）**。為了做到這一點，我們要定義一個自訂的模型評分函式！

> 🧩 **白話小補充：為什麼要「自訂」？** 上一課的 `llm-rubric` 只能給「通過／不通過」的判斷，評分標準只有一句話。如果你想要**多個面向分別打分數、算平均、自訂通過門檻、附上評分範例**，就需要自己寫評分函式。

---

## 輸入資料

我們的目標是寫一個提示詞，把複雜的維基百科文章摘要成簡短、好懂的摘要。我們先收集要在評估中摘要的文章。

在這個資料夾裡，我們提供了一個 `articles` 資料夾，裡面有 **8 個 txt 檔案**。每個檔案都包含一篇維基百科文章的文字內容。我們會把這些文章當作評估的輸入。打開幾個文章檔案看看，感受一下它們有多長、多複雜。

（8 篇文章的主題：大型語言模型、詞嵌入、反向傳播、卷積神經網路、手寫辨識、GPU（圖形處理器）、IBM Watson、自然語言處理。每篇都有一萬到六萬個字元。）

這個資料集只有 8 個測試案例，對於真實世界的評估來說太小了。就像我們在這門課中提過很多次的，我們強烈建議使用至少 100 筆資料的評估資料集。

---

## 我們的提示詞

看看 `prompts.py` 檔案。它包含三個不同的提示詞產生函式，我們會用 promptfoo 來評估它們：

```py
def basic_summarize(article):
  return f"Summarize this article {article}"

def better_summarize(article):
  return f"""
  Summarize this article for a grade-school audience: {article}"""

def best_summarize(article):
  return f"""
  You are tasked with summarizing long wikipedia articles for a grade-school audience.
  Write a short summary, keeping it as concise as possible. 
  The summary is intended for a non-technical, grade-school audience. 
  This is the article: {article}"""
```

**翻譯**：
- `basic_summarize`（基本）：「摘要這篇文章 {article}」
- `better_summarize`（較好）：「為小學生讀者摘要這篇文章：{article}」
- `best_summarize`（最好）：「你的任務是為小學生讀者摘要很長的維基百科文章。寫一份簡短的摘要，盡可能保持簡潔。這份摘要是給沒有技術背景的小學生看的。這是文章：{article}」

**要特別注意的是，這些提示詞整體來說都很普通。我們刻意讓提示詞保持簡短，沒有遵循最佳做法（例如加入完整的範例），以盡量減少執行這組評估時使用的 token 數量。**

---

## 更新設定檔

`promptfooconfig.yaml` 檔案包含的欄位，大部分我們之前都看過了：

```yaml
description: 'Summarization Evaluation'

prompts:
  - prompts.py:basic_summarize
  - prompts.py:better_summarize
  - prompts.py:best_summarize

providers:
  - id: anthropic:messages:claude-3-5-sonnet-20240620
    label: "3.5 Sonnet"

tests:
  - vars:
      article: file://articles/article1.txt
  - vars:
      article: file://articles/article2.txt
  - vars:
      article: file://articles/article3.txt
  - vars:
      article: file://articles/article4.txt
  - vars:
      article: file://articles/article5.txt
  - vars:
      article: file://articles/article6.txt
  - vars:
      article: file://articles/article7.txt
  - vars:
      article: file://articles/article8.txt

defaultTest:
  assert:
    - type: python
      value: file://custom_llm_eval.py

```

我們告訴 promptfoo：我們想使用 `prompts.py` 裡定義的三個提示詞。接著，我們設定 promptfoo 使用 Claude 3.5 Sonnet 作為供應商。

（`providers` 這次用了另一種寫法：`id` 是模型、`label` 是在報表上顯示的名稱。）

我們寫了一系列的 `tests`，每個測試都提供不同的 `article` 值。這裡**唯一新的東西**，是我們從文字檔載入這些值。文章太長了，直接寫在 YAML 檔裡不太合理。舉例來說，設定檔的這一段：

```yaml
tests:
  - vars:
      article: file://articles/article1.txt
```

告訴 promptfoo：我們想執行一個測試，其中 `article` 變數被設定成 `article1.txt` 檔案的文字內容。我們對全部 8 個文章檔案都重複這個做法。

---

## 撰寫自訂的模型評分函式

接下來，我們把注意力轉到 YAML 檔案的最後一個欄位：

```yaml
defaultTest:
  assert:
    - type: python
      value: file://custom_llm_eval.py
```

這個欄位告訴 promptfoo：對於每一個測試，我們都想執行我們在 `custom_llm_eval.py` 檔案裡定義的一個特定 Python 斷言。我們之前在定義自訂程式評分斷言時看過這個語法。**唯一的差別是，這次我們要寫的函式會用另一個模型來替模型的輸出評分。**

我們來看看 `custom_llm_eval.py` 檔案的內容。它包含相當多的程式碼：

```py
import anthropic
import os
import json

def llm_eval(summary, article):
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    prompt = f"""Evaluate the following summary based on these criteria:
    1. Conciseness (1-5) - is the summary as concise as possible?
        - Conciseness of 1: The summary is unnecessarily long, including excessive details, repetitions, or irrelevant information. It fails to distill the key points effectively.
        - Conciseness of 3:  The summary captures most key points but could be more focused. It may include some unnecessary details or slightly over explain certain concepts.
        - Conciseness of 5: The summary effectively condenses the main ideas into a brief, focused text. It includes all essential information without any superfluous details or explanations.
    2. Accuracy (1-5) - is the summary completely accurate based on the initial article'?
        - Accuracy of 1: The summary contains significant errors, misrepresentations, or omissions that fundamentally alter the meaning or key points of the original article.
        - Accuracy of 3:  The summary captures some key points correctly but may have minor inaccuracies or omissions. The overall message is generally correct, but some details may be wrong.
        - Accuracy of 5: The summary faithfully represents the main gist of the original article without any errors or misinterpretations. All included information is correct and aligns with the source material.
    3. Tone (1-5) - is the summary appropriate for a grade school student with no technical training?
        - Tone of 1: The summary uses language or concepts that are too complex, technical, or mature for a grade school audience. It may contain jargon, advanced terminology, or themes that are not suitable for young readers.
        - Tone of 2:  The summary mostly uses language suitable for grade school students but occasionally includes terms or concepts that may be challenging. Some explanations might be needed for full comprehension.
        - Tone of 3: The summary consistently uses simple, clear language that is easily understandable by grade school students. It explains complex ideas in a way that is accessible and engaging for young readers.
    4. Explanation - a general description of the way the summary is evaluated

    <examples>
    <example>
    This summary:
    <summary>
    Artificial neural networks are computer systems inspired by how the human brain works. They are made up of interconnected "neurons" that process information. These networks can learn to do tasks by looking at lots of examples, similar to how humans learn. 

    Some key things about neural networks:
    - They can recognize patterns and make predictions
    - They improve with more data and practice
    - They're used for things like identifying objects in images, translating languages, and playing games

    Neural networks are a powerful tool in artificial intelligence and are behind many of the "smart" technologies we use today. While they can do amazing things, they still aren't as complex or capable as the human brain.
    <summary>
    Should receive a 5 for tone, a 5 for accuracy, and a 5 for conciseness
    </example>

    <example>
    This summary:
    <summary>
    Here is a summary of the key points from the article on artificial neural networks (ANNs):

    1. ANNs are computational models inspired by biological neural networks in animal brains. They consist of interconnected artificial neurons that process and transmit signals.

    2. Basic structure:
    - Input layer receives data
    - Hidden layers process information 
    - Output layer produces results
    - Neurons are connected by weighted edges

    3. Learning process:
    - ANNs learn by adjusting connection weights
    - Use techniques like backpropagation to minimize errors
    - Can perform supervised, unsupervised, and reinforcement learning

    4. Key developments:
    - Convolutional neural networks (CNNs) for image processing
    - Recurrent neural networks (RNNs) for sequential data
    - Deep learning with many hidden layers

    5. Applications:
    - Pattern recognition, classification, regression
    - Computer vision, speech recognition, natural language processing
    - Game playing, robotics, financial modeling

    6. Advantages:
    - Can model complex non-linear relationships
    - Ability to learn and generalize from data
    - Adaptable to many different types of problems

    7. Challenges:
    - Require large amounts of training data
    - Can be computationally intensive
    - "Black box" nature can make interpretability difficult

    8. Recent advances:
    - Improved hardware (GPUs) enabling deeper networks
    - New architectures like transformers for language tasks
    - Progress in areas like generative AI

    The article provides a comprehensive overview of ANN concepts, history, types, applications, and ongoing research areas in this field of artificial intelligence and machine learning.
    </summary>
    Should receive a 1 for tone, a 5 for accuracy, and a 3 for conciseness
    </example>
    </examples>

    Provide a score for each criterion in JSON format. Here is the format you should follow always:

    <json>
    {{
    "conciseness": <number>,
    "accuracy": <number>,
    "tone": <number>,
    "explanation": <string>,
    }}
    </json>


    Original Text: <original_article>{article}</original_article>
    
    Summary to Evaluate: <summary>{summary}</summary>
    """
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=1000,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": "<json>" 
            }
        ],
        stop_sequences=["</json>"]
    )
    
    evaluation = json.loads(response.content[0].text)
    # Filter out non-numeric values and calculate the average
    numeric_values = [value for key, value in evaluation.items() if isinstance(value, (int, float))]
    avg_score = sum(numeric_values) / len(numeric_values)
    return avg_score, response.content[0].text

def get_assert(output: str, context, threshold=4.5):
    article = context['vars']['article']
    score, evaluation = llm_eval(output, article )
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }

```

（新手版資料夾裡的 `custom_llm_eval.py` 和上面幾乎一樣，只是評分模型換成了 `claude-haiku-4-5`。原課程的檔案裡，語氣是第「4」項、說明是第「5」項，講義這裡則寫成 3 和 4，只是編號的小差異，不影響效果。）

### `get_assert()`

這裡有很多可以討論的，我們先從檔案最下面的函式開始：`get_assert`

```py
def get_assert(output: str, context, threshold=4.5):
    article = context['vars']['article']
    score, evaluation = llm_eval(output, article )
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }
```

回想一下之前那一課：promptfoo 會自動在斷言檔案裡尋找一個叫做 `get_assert` 的函式。它會傳給這個函式下面兩個參數：

- 某個模型回答的 **`output`（輸出）**
- **`context` 字典**，包含產生這個輸出的變數和提示詞

promptfoo 預期我們的函式回傳下面其中一種：
- 一個布林值（通過／失敗）
- 一個浮點數（分數）
- 一個 GradingResult 字典

我們選擇回傳 GradingResult 字典，它必須包含下面這些屬性：

- `pass`：布林值
- `score`：浮點數
- `reason`：一段文字說明

下面是加上註解、說明在做什麼的版本：

```py
def get_assert(output: str, context, threshold=4.5):
    # Get the specific article from the context
    article = context['vars']['article']
    #Pass the model output and the article to a function we've defined called llm_eval
    score, evaluation = llm_eval(output, article ) #capture the resulting score it returns and the evaluation explanation
    #return a dictionary indicating whether the output passed the test, its score, and the explanation behind the score
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }
```

**註解翻譯：**
- 「從 context 取出這次的文章」
- 「把模型的輸出（摘要）和文章傳給我們定義的 `llm_eval` 函式」「接住它回傳的分數和評分說明」
- 「回傳一個字典，表示輸出是否通過測試、它的分數，以及分數背後的說明」

> 🧩 **白話小補充**：
> - `threshold=4.5`：參數的預設值，通過門檻是 4.5 分。
> - `score, evaluation = llm_eval(...)`：`llm_eval` 回傳**兩個值**，用兩個變數一次接住（叫做「解包」）。
> - `"pass": score >= threshold`：分數大於等於 4.5 就是 `True`（通過）。

### `llm_eval()`

接著，我們仔細看看真正在評分的 `llm_eval` 函式。這個函式做了下面這些事：

1. 定義一個**非常長的評分準則提示詞**，說明摘要應該怎麼評分
2. 送出請求到 Anthropic API，執行評分提示詞
3. 解析回答，計算**平均分數**
4. 回傳平均分數和模型的完整文字回答

（原課程在這裡把完整的 `llm_eval` 程式碼又列了一次，內容和上面的檔案相同，新手版不重複列出。函式開頭的說明文字〔docstring〕寫著：「用 LLM（Claude）評估摘要。參數：summary（字串）要評估的摘要；article（字串）被摘要的原文。回傳：如果平均分數高於門檻就是 True，否則是 False。」——不過實際上它回傳的是「平均分數和評分文字」，docstring 寫得不太準確。）

**評分提示詞的重點翻譯：**

> 根據下列標準評估這份摘要：
> 1. **簡潔度（1～5）**——摘要是不是已經盡可能簡潔？
>    - 1 分：摘要不必要地冗長，包含過多細節、重複或不相關的資訊，沒能有效提煉出重點。
>    - 3 分：摘要抓到大部分重點，但可以更聚焦。可能包含一些不必要的細節，或對某些概念解釋得有點過多。
>    - 5 分：摘要有效地把主要概念濃縮成簡短、聚焦的文字，包含所有必要的資訊，沒有任何多餘的細節或解釋。
> 2. **正確性（1～5）**——根據原文，摘要是不是完全正確？
>    - 1 分：摘要包含重大錯誤、曲解或遺漏，根本地改變了原文的意思或重點。
>    - 3 分：摘要正確抓到一些重點，但可能有小錯誤或遺漏。整體訊息大致正確，但有些細節可能是錯的。
>    - 5 分：摘要忠實呈現原文的主旨，沒有任何錯誤或誤解，所有資訊都正確並與原始資料一致。
> 3. **語氣（1～5）**——摘要是否適合沒有技術背景的小學生？
>    - 1 分：使用的語言或概念對小學生來說太複雜、太技術性或太成熟，可能包含行話、進階術語或不適合年幼讀者的主題。
>    - 2 分：大部分使用適合小學生的語言，但偶爾包含可能有挑戰性的詞彙或概念，可能需要一些解釋才能完全理解。
>    - 3 分：一貫使用小學生容易理解的簡單、清楚的語言，用平易近人、吸引年幼讀者的方式解釋複雜的概念。
> 4. **說明**——對這份摘要如何被評估的整體描述
>
> `<examples>`：兩個評分範例——
> - 第一份摘要（用簡單的話解釋人工神經網路，例如「受人腦啟發的電腦系統」、「透過看很多例子學習」）→ **語氣 5、正確性 5、簡潔度 5**
> - 第二份摘要（列了 8 大點、充滿「隱藏層」、「反向傳播」、「卷積神經網路」等專業術語）→ **語氣 1、正確性 5、簡潔度 3**
>
> 用 JSON 格式提供每個標準的分數，格式是：`<json>{"conciseness": 數字, "accuracy": 數字, "tone": 數字, "explanation": 文字}</json>`
>
> 原文：`<original_article>{article}</original_article>`
> 要評估的摘要：`<summary>{summary}</summary>`

> 🧩 **白話小補充**：
> - 你可能注意到「語氣」的評分說明只寫到 3 分，但標題寫 1～5 分。這是原課程的小瑕疵，評審模型通常還是會給出 1～5 的分數。
> - 第一個範例的 `<summary>` 結尾少了斜線（寫成 `<summary>` 而不是 `</summary>`），也是原課程的筆誤。
> - JSON 格式範例裡用了 `{{` 和 `}}`：因為整個提示詞是 f-string，要顯示「真正的大括號」就要寫兩個。

**程式碼的其他部分：**
- `messages` 裡最後一則是 `{"role": "assistant", "content": "<json>"}`：**prefill**，幫 Claude 先寫好 `<json>`，它就會直接從 JSON 內容開始寫（[第 2 單元第 5 章](../../02_prompt_engineering_tutorial/05_Formatting_Output_and_Speaking_for_Claude.md)）。
- `stop_sequences=["</json>"]`：寫到 `</json>` 就停下，所以回答裡**只有 JSON 本身**。
- `json.loads(response.content[0].text)`：把 JSON 文字轉成字典。
- `[value for key, value in evaluation.items() if isinstance(value, (int, float))]`：只挑出**數字**的值（三個分數），排除 `explanation` 那段文字。（`isinstance(值, (int, float))` 是「檢查這個值是不是整數或小數」。）
- `avg_score = sum(...) / len(...)`：算平均分數。
- `temperature=0`：讓評分盡量穩定一致。

---

## 執行評估

我們用之前看過的同一個指令執行評估：

```bash
npx promptfoo@latest eval
```

這個過程可能要花一點時間才會完成，因為我們要先送出請求，讓模型產生文章摘要，然後再送出額外的請求來替這些摘要評分！

這是我們得到的評估結果截圖：

![評估結果](../../../prompt_evaluations/09_custom_model_graded_prompt_foo/images/eval_result.png)

我們啟動網頁畫面，更了解結果：

```bash
npx promptfoo@latest view
```

這是網頁儀表板的截圖：

![網頁儀表板](../../../prompt_evaluations/09_custom_model_graded_prompt_foo/images/web_view.png)

我們可以點擊每個格子裡的放大鏡，查看測試結果的更多資訊：

![評分說明](../../../prompt_evaluations/09_custom_model_graded_prompt_foo/images/explanation.png)

我們可以看到，這個特定的輸出沒有通過我們的自訂 llm-eval 函式，因為它的**語氣分數太低**。

另外，結果的最上面一列顯示了每個提示詞的分數摘要：

![每個提示詞的整體分數](../../../prompt_evaluations/09_custom_model_graded_prompt_foo/images/overall_scores.png)

不出所料，我們的 `best_summary` 提示詞表現最好！

儀表板的上方也顯示了一些圖表，幫助視覺化分數：

![分數分布圖](../../../prompt_evaluations/09_custom_model_graded_prompt_foo/images/distribution.png)

在上面的截圖中：

* **紅色**是 `basic_summarize` 提示詞
* **藍色**是 `better_summarize` 提示詞
* **綠色**是 `best_summarize` 提示詞

這張圖告訴我們：`best_summarize` 提示詞不只**從來沒有**沒通過測試，它在**所有**輸入上的分數都比其他提示詞高。

---

## 📌 重點整理

- **自訂模型評分** = 在 `get_assert()` 裡呼叫 Claude，用你設計的評分提示詞打分數。
- 好的評分提示詞要有：**多個明確的標準**、**每個分數等級的描述**、**評分範例**、**固定的輸出格式（JSON）**。
- 用 **prefill `<json>` + `stop_sequences=["</json>"]`** 讓評審只輸出 JSON，方便 `json.loads()` 解析。
- 回傳 `{"pass": 平均 >= 門檻, "score": 平均, "reason": 評分說明}`，promptfoo 會顯示在報表上。
- 很長的測試資料可以用 `file://路徑` 從檔案載入。

🎉 **恭喜你完成第 4 單元「提示詞評估」！** 你已經學會人工評分、程式評分、模型評分，以及用 promptfoo 有系統地比較提示詞和模型。

👉 下一個單元：[第 5 單元：工具使用 — 5-1 工具使用概觀](../../05_tool_use/01_tool_use_overview.md)